# Week 03 · Feature engineering · Lab

Last week ended with an uncomfortable result: judged on the future, Week 1's baselines were barely better than guessing. This week we build features that do better, and we build them so they cannot leak.

**How this lab works**

- **You write most of the code.** Cells marked **TODO** are yours, most with hints.
- Cells marked **Check** test your work, and tell you where to look if something is off.
- After key results there is a question for you to answer in your own words.
- It is longer than one session **on purpose**. Parts 5 to 7 make good homework.
- You may use an AI assistant for code. You may not skip understanding what it wrote.
- Work in your own copy: *File → Save As* `my-lab.ipynb`.

## Setup

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, "..")
from checkpoints import week_02

RANDOM_STATE = 42
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})
ACCENT, WARM, GREY = "#3f51b5", "#e07b39", "#c9cdd6"


def lift(y_true, scores):
    """PR AUC divided by the positive rate: how many times better than guessing."""
    return average_precision_score(y_true, scores) / np.mean(y_true)

## 0. Start where we left off

We load last week's clean table, one row per order, already split by time.

In [ ]:
df = week_02()
train, valid, test = (df[df["split"] == part].copy() for part in ["train", "valid", "test"])
print(f"train {len(train):,}   valid {len(valid):,}   test {len(test):,}")

**The bar to beat.** In Week 2, Week 1's rule and tree scored a lift of **1.57 and 1.65 on validation**, and **1.10 and 1.18 on test**. Every choice this week is made on validation. Test we look at once, at the very end.

## 1. Two kinds of feature

Every feature you will ever build falls into one of two kinds, and the difference decides whether it can leak.

| Kind | Computed from | Example | When it is safe to build |
|---|---|---|---|
| **Row-wise** | One row, on its own | Distance between seller and customer | Anytime. The answer is the same however you split the data. |
| **Learned** | Many rows | A seller's average late rate | Only from training rows, after the split |

A row-wise feature gives the same value for an order whether it sits in train or test. A learned feature does not: it depends on which other rows were used to compute it. That is exactly where leaks come from.

We start with the safe kind.

### Row-wise features

Last week's charts suggested distance and same state. Two more come straight from the data: how many days were promised, and how large the freight cost is compared with the price.

For distance we use the **haversine** formula, the straight-line distance between two points on a sphere:

In [ ]:
# TODO: write haversine_km. It takes latitudes and longitudes in degrees and returns kilometres.
#       1. convert all four to radians:  lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
#       2. a = sin²((lat2 - lat1) / 2) + cos(lat1) · cos(lat2) · sin²((lng2 - lng1) / 2)
#       3. distance = 6371 · 2 · arcsin(√a)        (6371 km is the Earth's radius)
#       Use np.sin, np.cos, np.arcsin, np.sqrt so it works on whole columns at once.
def haversine_km(lat1, lng1, lat2, lng2):
    ...

print(f"São Paulo to Rio de Janeiro: {haversine_km(-23.55, -46.63, -22.91, -43.17):.0f} km")

In [ ]:
# Check
assert abs(haversine_km(-23.55, -46.63, -22.91, -43.17) - 360.6) < 1, "São Paulo to Rio should be about 361 km. Did you convert to radians first?"
print("Correct.")

Now the four features. Each one uses a single row and nothing else, so we can add them to the whole table at once.

In [ ]:
# TODO: add four columns to a copy of `frame`, then return it
#       distance_km    haversine_km from the seller's coordinates to the customer's
#       same_state     1 if customer_state equals seller_state, else 0
#       window_days    days from the purchase day to the promised date (as in Week 1)
#       freight_ratio  total_freight divided by total_price
def add_row_wise(frame):
    frame = frame.copy()
    ...
    return frame

train, valid, test = add_row_wise(train), add_row_wise(valid), add_row_wise(test)
train[["distance_km", "same_state", "window_days", "freight_ratio"]].describe().round(2)

In [ ]:
# Check
assert abs(train["distance_km"].median() - 453.3) < 1, "Median distance in train should be about 453 km."
assert abs(train["same_state"].mean() - 0.335) < 0.002, "About a third of training orders stay within one state."
assert train["window_days"].median() == 25, "window_days: promised date minus the normalised purchase day, in days."
assert abs(train["freight_ratio"].median() - 0.224) < 0.002, "freight_ratio is total_freight / total_price."
print("Correct.")

The typical order travels about 450 km, a third stay within one state, and freight costs about a fifth of the price.

### Which of them carry signal?

A quick test before any model: use each feature, alone, as a score, and measure its lift on validation. Some features predict lateness when they are *high*, others when they are *low*, so we try both directions and keep the better one.

In [ ]:
ROW_WISE = ["distance_km", "same_state", "window_days", "freight_ratio", "total_freight", "total_weight_g", "n_items", "total_price"]

single = {}
for feature in ROW_WISE:
    values = valid[feature].fillna(train[feature].median())
    high, low = lift(valid["is_late"], values), lift(valid["is_late"], -values)
    single[feature] = pd.Series({"lift": max(high, low), "late when": "high" if high >= low else "low"})
single = pd.DataFrame(single).T.sort_values("lift", ascending=False)

ax = single["lift"].astype(float).sort_values().plot.barh(color=ACCENT, figsize=(7, 3.8))
ax.axvline(1, color="#333333", ls="--", lw=1)
ax.set_xlabel("lift on validation, feature used alone")
ax.set_title("How much does each feature say on its own?")
plt.show()
single

**Your turn:** which features look useful on their own, and which look useless? Does "useless alone" mean "useless in a model"?

*Your answer:* ...

## 2. The leak Week 1 promised

In Week 1 we listed three kinds of leak, from obvious to quiet. The quietest was **a statistic computed over the whole dataset**. Here it is.

Some sellers are much more often late than others. So a seller's past late rate sounds like an excellent feature. The question is: computed from which orders?

In [ ]:
overall = train["is_late"].mean()

# TODO: each seller's late rate, computed two ways
#       rate_all    from every order in df, including validation and test
#       rate_train  from training orders only
#       Hint: groupby("seller_id")["is_late"].mean()
rate_all = ...
rate_train = ...

# TODO: map each validation order's seller to its rate. Sellers never seen get `overall`.
#       Hint: valid["seller_id"].map(...).fillna(overall)
leaky = ...
honest = ...

print(f"seller late rate from ALL data:   validation lift {lift(valid['is_late'], leaky):.2f}")
print(f"seller late rate from train only: validation lift {lift(valid['is_late'], honest):.2f}")

In [ ]:
# Check
assert abs(lift(valid["is_late"], leaky) - 1.89) < 0.02, "rate_all should use every order in df."
assert abs(lift(valid["is_late"], honest) - 1.21) < 0.02, "rate_train should use training orders only."
print("Correct.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(["computed on train only", "computed on all data"],
        [lift(valid["is_late"], honest), lift(valid["is_late"], leaky)], color=[ACCENT, WARM], height=0.55)
ax.axvline(1, color="#333333", ls="--", lw=1)
ax.set_xlabel("validation lift")
ax.set_title("Same feature, same data, one leak")
plt.show()

**Your turn:** explain in your own words why the "all data" version scores higher. Where exactly is the answer hiding?

*Your answer:* ...

## 3. Your own transformer

The fix for learned features is a pattern, not a trick. A **transformer** has two methods:

- `fit(X, y)` **learns** something from the training data, and stores it
- `transform(X)` **applies** what was learned, to any data

Learning happens once, on train. Applying happens everywhere. This is the same pattern as the `MedianImputer` class in the [Python refresher](https://evisp.github.io/ml-course-umt/01-toolkit/python-refresher/), and every preprocessing step in scikit-learn follows it.

Let's build one. Last week's promised-versus-actual chart showed that Olist pads its promises. So a useful question is: **how much slack does this promise leave, compared with how long this route usually takes?** The typical time per route must be learned, so it goes in `fit`. The slack is computed in `transform`.

One detail matters here. `fit` may look at the actual delivery dates, because it only ever sees training orders, which are in the past. `transform` must **not**: at the moment of purchase, the delivery has not happened yet.

In [ ]:
class RouteSlack(BaseEstimator, TransformerMixin):
    """Days of slack in the promise, compared with the typical delivery time on this route."""

    def fit(self, X, y=None):
        # TODO: learn the typical delivery time per route, from X only
        #       1. days_taken: delivered day minus purchase day, in days (normalise both)
        #       2. route: seller_state + " > " + customer_state
        #       3. store the median days_taken per route in self.typical_days_
        #          and the overall median in self.overall_days_
        ...
        return self

    def transform(self, X):
        # TODO: use what fit learned. Never read the delivery date here.
        #       1. route, as in fit
        #       2. typical = route.map(self.typical_days_).fillna(self.overall_days_)
        #       3. promised = days from the purchase day to the promised date
        #       4. return pd.DataFrame({"slack_days": promised - typical}, index=X.index)
        ...

    def get_feature_names_out(self, input_features=None):
        return np.array(["slack_days"])

Fit on train, apply to validation:

In [ ]:
slack = RouteSlack().fit(train)

print(f"routes learned: {len(slack.typical_days_)}")
print(f"typical São Paulo to Rio de Janeiro: {slack.typical_days_['SP > RJ']:.0f} days")
slack.transform(valid).describe().round(1)

In [ ]:
# Check
assert len(slack.typical_days_) == 369, "fit should store one median per route seen in training: 369 routes."
assert slack.typical_days_["SP > RJ"] == 13, "The typical São Paulo to Rio trip in training is 13 days."
print("Correct.")

A typical São Paulo to Rio parcel takes 13 days, and the median validation order is promised 11 days more than its route usually needs. Olist's padding, in one number per order.

Notice that the transformer holds what it learned in attributes ending in an underscore: `typical_days_`, `overall_days_`. That is the scikit-learn convention for "learned during fit".

## 4. The pipeline

Doing each step by hand, as in Part 2, is how leaks creep in: one forgotten `fit` on the wrong data is enough. A **pipeline** makes it structurally impossible. It holds every preprocessing step and the model in one object. Calling `fit` on it fits every step on the data you pass, in order. Calling `predict` applies them all, using only what was learned during fit.

Different columns need different treatment, so we use a `ColumnTransformer` to route them:

| Columns | Treatment |
|---|---|
| The eight row-wise numbers | Fill gaps with the training median, then scale |
| Customer and seller state | One-hot encode: 27 states each, a manageable number of columns |

In [ ]:
# TODO: a small pipeline for the numeric columns: SimpleImputer(strategy="median"), then StandardScaler()
numeric = ...

# TODO: a ColumnTransformer with two parts
#       ("numeric", numeric, ROW_WISE)
#       ("states", OneHotEncoder(handle_unknown="ignore"), ["customer_state", "seller_state"])
prep = ...

# TODO: the full pipeline: ("prep", prep), then ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
model = ...

model.fit(train, train["is_late"])
print(f"validation lift: {lift(valid['is_late'], model.predict_proba(valid)[:, 1]):.2f}")

In [ ]:
# Check
assert abs(lift(valid["is_late"], model.predict_proba(valid)[:, 1]) - 2.34) < 0.02, "Validation lift should be about 2.34. Check both parts of the ColumnTransformer."
print("Correct.")

**Your turn:** the imputer fills gaps with a median. Which rows was that median computed from, and why does it matter?

*Your answer:* ...

## 5. Target encoding, naive and right

States have 27 values each, so one-hot encoding is fine. Sellers have almost three thousand. One-hot would create three thousand columns, most of them nearly empty.

**Target encoding** replaces each seller with a single number: its late rate. It is a learned feature, so it must come from training rows. But there is a second, subtler trap. If each training order is encoded using a rate that *includes its own label*, the model learns to trust the feature far too much.

Let's see it. First the naive way: compute each seller's rate on the training data, and use it to encode those same training rows.

In [ ]:
def with_seller(frame, rates):
    return frame.assign(seller_rate=frame["seller_id"].map(rates).fillna(overall))

def seller_model():
    prep = ColumnTransformer([
        ("numeric", numeric, ROW_WISE + ["seller_rate"]),
        ("states", OneHotEncoder(handle_unknown="ignore"), ["customer_state", "seller_state"]),
    ])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])

# TODO: encode train and validation with rate_train (from Part 2), using with_seller
naive_train = ...
naive_valid = ...

# TODO: fit seller_model() on naive_train
naive = ...

print(f"naive encoding: train lift {lift(train['is_late'], naive.predict_proba(naive_train)[:, 1]):.2f}, "
      f"validation lift {lift(valid['is_late'], naive.predict_proba(naive_valid)[:, 1]):.2f}")

In [ ]:
# Check
assert lift(train["is_late"], naive.predict_proba(naive_train)[:, 1]) > 3.5, "Encode the training rows with rate_train, computed from those same rows."
print("Correct.")

A training lift near 4 and a validation lift far below it. The model has learned that `seller_rate` is almost the answer, because on training rows it partly *is*: each row's own label is inside its seller's average. On new orders, it is not.

The right way is **cross-fitting**. The training rows are split into folds, and each row is encoded using a rate computed from the *other* folds only, so no row ever sees its own label. scikit-learn's `TargetEncoder` does exactly this inside `fit_transform`.

In [ ]:
encoder = TargetEncoder(random_state=RANDOM_STATE)
fair_train = train.assign(seller_rate=encoder.fit_transform(train[["seller_id"]], train["is_late"]).ravel())
fair_valid = valid.assign(seller_rate=encoder.transform(valid[["seller_id"]]).ravel())
fair = seller_model().fit(fair_train, train["is_late"])

comparison = pd.DataFrame({
    "train": [lift(train["is_late"], naive.predict_proba(naive_train)[:, 1]),
              lift(train["is_late"], fair.predict_proba(fair_train)[:, 1])],
    "validation": [lift(valid["is_late"], naive.predict_proba(naive_valid)[:, 1]),
                   lift(valid["is_late"], fair.predict_proba(fair_valid)[:, 1])],
}, index=["naive", "cross-fitted"])

ax = comparison.plot.bar(color=[GREY, ACCENT], rot=0, figsize=(6, 3.6), width=0.6)
ax.set_ylabel("lift")
ax.set_title("Naive encoding looks brilliant, until new data arrives")
plt.show()
comparison.round(2)

**Your turn:** why does the naive version score so well on train, and so poorly on validation?

*Your answer:* ...

## 6. Why logistic regression needs scaling

Our numeric features live on very different scales: distance in hundreds of kilometres, same state as 0 or 1, freight ratio around 0.2. Logistic regression learns by gradient steps, and features on wildly different scales make those steps inefficient. Remove the scaler and see.

In [ ]:
def fit_and_report(scale, label):
    steps = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale else [])
    prep = ColumnTransformer([
        ("numeric", Pipeline(steps), ROW_WISE),
        ("states", OneHotEncoder(handle_unknown="ignore"), ["customer_state", "seller_state"]),
    ])
    pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=300, class_weight="balanced"))])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pipe.fit(train, train["is_late"])
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    print(f"{label:16} converged: {converged!s:6} validation lift: {lift(valid['is_late'], pipe.predict_proba(valid)[:, 1]):.2f}")

fit_and_report(scale=True, label="with scaling")
fit_and_report(scale=False, label="without scaling")

Without scaling, the optimiser runs out of steps before it finds the best weights. You get a model, and a warning that is easy to ignore.

Scaling has a second benefit: once every feature has the same scale, the **coefficients become comparable**. A larger coefficient means a larger effect for a typical change in that feature.

In [ ]:
names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["model"].coef_[0], index=names)

# TODO: keep only the eight numeric coefficients, strip the "numeric__" prefix, sort them,
#       and draw a horizontal bar chart. Colour positive bars WARM and negative ones ACCENT,
#       and add a vertical line at 0.
#       Hint: coefs[[f"numeric__{f}" for f in ROW_WISE]].rename(lambda n: n.replace("numeric__", ""))

**Your turn:** which two features push hardest in each direction? Do they agree with Week 2's charts?

*Your answer:* ...

## 7. A feature must earn its place

We now have several candidate additions to the row-wise model: the route slack from Part 3, the category, and the seller. Each sounds reasonable. The only fair way to decide is to add each one to the base model **on its own**, and judge it **on validation**.

In [ ]:
def build(extra=None, estimator=None):
    parts = [
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), ROW_WISE),
        ("states", OneHotEncoder(handle_unknown="ignore"), ["customer_state", "seller_state"]),
    ]
    if extra == "route slack":
        route_columns = ["order_purchase_timestamp", "order_estimated_delivery_date",
                         "order_delivered_customer_date", "seller_state", "customer_state"]
        parts.append(("slack", Pipeline([("slack", RouteSlack()), ("scale", StandardScaler())]), route_columns))
    elif extra == "category":
        parts.append(("category", TargetEncoder(random_state=RANDOM_STATE), ["category"]))
    elif extra == "seller":
        parts.append(("seller", TargetEncoder(random_state=RANDOM_STATE), ["seller_id"]))
    estimator = estimator or LogisticRegression(max_iter=2000, class_weight="balanced")
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", estimator)])

In [ ]:
candidates = {
    "row-wise only": build(),
    "+ route slack": build("route slack"),
    "+ category": build("category"),
    "+ seller": build("seller"),
    "row-wise, as a tree": build(estimator=DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE)),
}

# TODO: fit each pipeline on train, and store its validation lift in a dict called `scores`.
#       Then turn it into a pd.Series and draw a horizontal bar chart, best at the top.
scores = ...

In [ ]:
# Check
assert isinstance(scores, pd.Series) and len(scores) == 5, "scores should be a Series with one entry per candidate."
assert scores.idxmax() in ("row-wise only", "+ category"), "Fit each candidate on train and score it on validation."
print("Correct.")

None of the learned additions clearly beats the plain row-wise model. The category is a tie at best, and the route slack and the seller both make validation worse. The tree, on exactly the same features, loses too. Why trees behave this way on data like ours is a question for Block 2.

This is a result worth being proud of, not disappointed by. **Simpler won, and we know it because we tested it on data the model never saw**, rather than because it felt clever. So the chosen model is the row-wise logistic regression from Part 4.

### One look at test

Every decision has been made. Now, and only now, we score the chosen model on test, once.

In [ ]:
final_lift = lift(test["is_late"], model.predict_proba(test)[:, 1])
print(f"test lift: {final_lift:.2f}")

progress = pd.Series({
    "Week 1 rule": 1.10,
    "Week 1 tree": 1.18,
    "Week 3 model": final_lift,
})
ax = progress.plot.bar(color=[GREY, GREY, ACCENT], rot=0, figsize=(6, 3.6), width=0.6)
ax.axhline(1, color="#333333", ls="--", lw=1)
for i, value in enumerate(progress):
    ax.text(i, value + 0.03, f"{value:.2f}", ha="center")
ax.set_ylabel("lift on test")
ax.set_ylim(0, 2.2)
ax.set_title("Judged on the future, once")
plt.show()

**Your turn:** the test lift is lower than the validation lift. Is that a sign something went wrong? Think back to Week 2.

*Your answer:* ...

## 8. Ready for next week

Week 4 evaluates this exact model in depth: where it fails, for whom, and at which threshold. The checkpoint module now has two functions for that. Let's confirm they match what we built.

In [ ]:
from checkpoints import week_03, week_03_pipeline

check = week_03()
check_train = check[check["split"] == "train"]
check_valid = check[check["split"] == "valid"]

same_features = np.allclose(check_train["distance_km"].fillna(-1), train["distance_km"].fillna(-1))
chosen = week_03_pipeline().fit(check_train, check_train["is_late"])
same_score = abs(lift(check_valid["is_late"], chosen.predict_proba(check_valid)[:, 1])
                 - lift(valid["is_late"], model.predict_proba(valid)[:, 1])) < 1e-9

print("same features:", same_features)
print("same model:   ", same_score)

## What we learned

Write, in one sentence each:

1. What is the difference between a row-wise feature and a learned one, and why does it matter?
2. Why did the seller rate computed on all the data score higher on validation?
3. What does cross-fitting prevent?
4. The learned features lost to the simple ones. How do you know that is true, and not bad luck?